# TD — pandas (bases)

L'analyse de journaux réseau se fait avec `pandas`, la bibliothèque standard
de manipulation de données en Python. Avant les TP « détection de beacon » et
« voyage impossible », ce TD installe les **bases** : charger un CSV, inspecter
un DataFrame, sélectionner, filtrer, transformer, joindre et agréger.

**Consignes :** complétez chaque fonction ou variable (corps entre `#BEGIN` / `#END`), puis
exécutez la cellule de tests associée. Les tests affichent `… OK` quand tout
passe.


In [1]:
import pandas as pd


## Exercice 1 — Charger les données (`read_csv`)

Pour charger des données, on appelle `pd.read_csv`. Le paramètre `parse_dates` convertit
la colonne `timestamp` en vraies dates, ce qui permet ensuite d'extraire
l'heure ou de calculer des écarts.

- Doc : https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html



In [4]:
connexions = pd.read_csv("connexions.csv", parse_dates=["timestamp"])

La cellule suivante affiche un aperçu de ce qu'on vient de charger : les
dimensions (`.shape`), le type de chaque colonne (`.info()`) et les premières
lignes (`.head()`).


In [5]:
print("shape :", connexions.shape)
print("\n-------\n")
connexions.info()
print("\n-------\n")
connexions.head()


shape : (1002, 3)

-------

<class 'pandas.DataFrame'>
RangeIndex: 1002 entries, 0 to 1001
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   timestamp  1002 non-null   datetime64[us]
 1   username   1002 non-null   str           
 2   ip         1002 non-null   str           
dtypes: datetime64[us](1), str(2)
memory usage: 23.6 KB

-------



,timestamp,username,ip
0,2026-10-07 08:00:00,alice.dupont,192.0.2.48
1,2026-10-07 18:00:00,alice.dupont,192.0.2.48
2,2026-10-08 08:00:00,alice.dupont,192.0.2.48
3,2026-10-08 18:00:00,alice.dupont,192.0.2.48
4,2026-10-09 08:00:00,alice.dupont,192.0.2.48


In [6]:
connexions

,timestamp,username,ip
0,2026-10-07 08:00:00,alice.dupont,192.0.2.48
1,2026-10-07 18:00:00,alice.dupont,192.0.2.48
2,2026-10-08 08:00:00,alice.dupont,192.0.2.48
3,2026-10-08 18:00:00,alice.dupont,192.0.2.48
4,2026-10-09 08:00:00,alice.dupont,192.0.2.48
...,...,...,...
997,2026-10-12 08:15:00,vincent.dubois,203.0.113.10
998,2026-10-12 18:00:00,vincent.dubois,192.0.2.39
999,2026-10-13 08:00:00,vincent.dubois,192.0.2.39
1000,2026-10-13 08:15:00,vincent.dubois,203.0.113.10


## Exercice 2 — Sélectionner une colonne

Une colonne se sélectionne avec `df["nom"]` : on obtient une **Series**, la
colonne du DataFrame (l'équivalent d'une liste indexée). C'est l'opération de
base pour isoler un champ à analyser.

- `df["col"]` : la colonne `col` (une `Series`).

Complétez la variable `ips` ci-dessous : elle contient la colonne `ip`.


In [8]:
ips = connexions[ "ip" ]
ips

0         192.0.2.48
1         192.0.2.48
2         192.0.2.48
3         192.0.2.48
4         192.0.2.48
            ...     
997     203.0.113.10
998       192.0.2.39
999       192.0.2.39
1000    203.0.113.10
1001      192.0.2.39
Name: ip, Length: 1002, dtype: str

In [9]:
assert isinstance(ips, pd.Series), "le résultat doit être une Series"
assert ips.nunique() == 55, f"attendu 55 IP distinctes, obtenu {ips.nunique()}"
print("ex 2 OK :", ips.nunique(), "IP distinctes")


ex 2 OK : 55 IP distinctes


## Exercice 3 — Sélectionner plusieurs colonnes

Pour isoler un sous-ensemble de colonnes, on passe une **liste** de noms entre
crochets : on obtient un **DataFrame** (et non une Series), limité aux colonnes
demandées.

- `df[["col1", "col2"]]` : un sous-DataFrame ne contenant que `col1` et `col2`.

Complétez la variable `identite` ci-dessous : elle contient les colonnes
`username` et `ip`.


In [10]:
identite = connexions[ ["username", "ip"] ]
identite

,username,ip
0,alice.dupont,192.0.2.48
1,alice.dupont,192.0.2.48
2,alice.dupont,192.0.2.48
3,alice.dupont,192.0.2.48
4,alice.dupont,192.0.2.48
...,...,...
997,vincent.dubois,203.0.113.10
998,vincent.dubois,192.0.2.39
999,vincent.dubois,192.0.2.39
1000,vincent.dubois,203.0.113.10


In [11]:
assert isinstance(identite, pd.DataFrame), "le résultat doit être un DataFrame (pas une Series)"
assert list(identite.columns) == ["username", "ip"], f"colonnes inattendues : {list(identite.columns)}"
print("ex 3 OK :", identite.shape[0], "lignes,", list(identite.columns))


ex 3 OK : 1002 lignes, ['username', 'ip']


## Exercice 4 — Filtrer les lignes

Pour ne garder que certaines lignes, on construit un **masque booléen** : une
Series de `True`/`False` (une condition sur une colonne), qu'on passe entre
crochets. Seules les lignes où le masque vaut `True` sont conservées.

- `df[df["col"] == val]` : les lignes du dataframe dont les valeurs de la colonne `col` sont égales à `val`
- Combiner avec `&` (et), `|` (ou), `~` (non) — les parenthèses sont
  obligatoires.

Extraire les connexions rattachées à `alice.dupont`.


In [17]:
conn_alice = connexions[
    (connexions["username"] == "alice.dupont")  
]
conn_alice

,timestamp,username,ip
0,2026-10-07 08:00:00,alice.dupont,192.0.2.48
1,2026-10-07 18:00:00,alice.dupont,192.0.2.48
2,2026-10-08 08:00:00,alice.dupont,192.0.2.48
3,2026-10-08 18:00:00,alice.dupont,192.0.2.48
4,2026-10-09 08:00:00,alice.dupont,192.0.2.48
5,2026-10-09 18:00:00,alice.dupont,192.0.2.48
6,2026-10-10 08:00:00,alice.dupont,192.0.2.48
7,2026-10-10 08:20:00,alice.dupont,192.0.2.49
8,2026-10-10 18:00:00,alice.dupont,192.0.2.48
9,2026-10-11 08:00:00,alice.dupont,192.0.2.48


In [18]:
assert len(conn_alice) == 15, f"attendu 15 connexions, obtenu {len(conn_alice)}"
assert (conn_alice["username"] == "alice.dupont").all()
print("ex 4 OK :", len(conn_alice), "connexions pour alice.dupont")


ex 4 OK : 15 connexions pour alice.dupont


In [23]:
connexions["timestamp"].dt.hour

0        8
1       18
2        8
3       18
4        8
        ..
997      8
998     18
999      8
1000     8
1001    18
Name: timestamp, Length: 1002, dtype: int32

## Exercice 5 — Assigner une nouvelle colonne

On ajoute une colonne en écrivant `df["nouvelle"] = ...`. La valeur peut être
constante ou calculée à partir des colonnes existantes. Ici, on extrait l'heure
de `timestamp` grâce à l'accesseur `.dt`, propre aux colonnes de dates.

- `df["nouvelle"] = ...` : ajoute (ou remplace) une colonne.
- `df["timestamp"].dt.hour` : l'heure (0-23) de chaque date.
- `.dt.date`, `.dt.minute`, `.dt.dayofweek`… : les autres composants.

Complétez la fonction `ajouter_heure()` ci-dessous.


In [24]:
def ajouter_heure(df: pd.DataFrame) -> pd.DataFrame:
    """Renvoie une copie de `df` avec une colonne `heure` (entier 0-23)."""
    res = df.copy()
    res["heure"] = res["timestamp"].dt.hour
    return res


In [25]:
avec_heures = ajouter_heure(connexions)
assert "heure" in avec_heures.columns, "colonne `heure` absente"
assert set(avec_heures["heure"].unique()) == {8, 9, 13, 18, 19, 20}
print("ex 5 OK : heures présentes =", sorted(avec_heures["heure"].unique()))


ex 5 OK : heures présentes = [np.int32(8), np.int32(9), np.int32(13), np.int32(18), np.int32(19), np.int32(20)]


## Exercice 6 — Transformer avec `.map`

Un journal ne contient souvent que des identifiants (ici, une IP). Pour les
traduire en information lisible, on utilise un **dictionnaire de correspondance**
puis `.map`, qui remplace chaque valeur par son image.

On charge ici une petite base de géolocalisation `ip_geo.csv` (IP → pays,
ville, coordonnées) :


In [26]:
geo = pd.read_csv("ip_geo.csv")
geo.head()

,ip,latitude,longitude,city,country
0,192.0.2.1,48.8566,2.3522,Paris,France
1,192.0.2.10,43.7102,7.2620,Nice,France
2,192.0.2.11,45.1885,5.7245,Grenoble,France
3,192.0.2.12,47.3220,5.0415,Dijon,France
4,192.0.2.13,48.8566,2.3522,Paris,France


In [39]:
connexions.head()

,timestamp,username,ip
0,2026-10-07 08:00:00,alice.dupont,192.0.2.48
1,2026-10-07 18:00:00,alice.dupont,192.0.2.48
2,2026-10-08 08:00:00,alice.dupont,192.0.2.48
3,2026-10-08 18:00:00,alice.dupont,192.0.2.48
4,2026-10-09 08:00:00,alice.dupont,192.0.2.48


- `colonne.map(dico)` : remplace chaque valeur par `dico[valeur]` (ou `NaN` si
  la clé est absente).
- `dict(zip(geo["ip"], geo["country"]))` : fabrique le dictionnaire ip → pays.

Complétez les deux fonctions ci-dessous :

1. `dico_pays()` construit le dictionnaire `ip → pays` depuis `geo` ;
2. `pays_de()` ajoute la colonne `pays` à `df` en s'appuyant sur ce dictionnaire.

In [43]:
def dico_pays(geo: pd.DataFrame) -> dict[str, str]:
    """Renvoie le dictionnaire ip → pays construit depuis `geo`."""
    ips = geo["ip"]
    countries = geo["country"]
    return dict(zip(ips, countries))

def pays_de(df: pd.DataFrame, geo: pd.DataFrame) -> pd.DataFrame:
    """Renvoie une copie de `df` avec une colonne `pays` (IP mappée vers son pays)."""
    res = df.copy()
    dic_pays = dico_pays(geo)
    res["pays"] = connexions["ip"].map(dic_pays)
    return res


In [44]:
# dic_pays = dico_pays(geo)
# connexions["ip"].map(dic_pays)

In [45]:
d = dico_pays(geo)
assert d["192.0.2.48"] == "France"
avec_pays = pays_de(connexions, geo)
assert "pays" in avec_pays.columns
assert not avec_pays["pays"].isna().any(), "des IP n'ont pas de pays"
assert (avec_pays[avec_pays["ip"] == "192.0.2.48"]["pays"] == "France").all()
print("ex 6 OK :", len(avec_pays["pays"].unique()), "pays distincts")


ex 6 OK : 5 pays distincts


In [46]:
avec_pays

,timestamp,username,ip,pays
0,2026-10-07 08:00:00,alice.dupont,192.0.2.48,France
1,2026-10-07 18:00:00,alice.dupont,192.0.2.48,France
2,2026-10-08 08:00:00,alice.dupont,192.0.2.48,France
3,2026-10-08 18:00:00,alice.dupont,192.0.2.48,France
4,2026-10-09 08:00:00,alice.dupont,192.0.2.48,France
...,...,...,...,...
997,2026-10-12 08:15:00,vincent.dubois,203.0.113.10,France
998,2026-10-12 18:00:00,vincent.dubois,192.0.2.39,France
999,2026-10-13 08:00:00,vincent.dubois,192.0.2.39,France
1000,2026-10-13 08:15:00,vincent.dubois,203.0.113.10,France


## Exercice 7 — Transformer avec `.apply`

`.map` remplace par une valeur issue d'un dictionnaire. Quand la transformation
est **calculée** (une fonction, une règle), on utilise `.apply`, qui applique
une fonction à chaque valeur de la colonne.

On classe chaque connexion selon son heure avec la fonction `categoriser_heure`
(fournie) :


In [47]:
def categoriser_heure(h: int) -> str:
    """Classe une heure : "nuit" (<8), "travail" (8-18), "soir" (>18)."""
    if h < 8:
        return "nuit"
    if h <= 18:
        return "travail"
    return "soir"


- `colonne.apply(f)` : applique `f` à chaque valeur, renvoie une Series.
- Pour une fonction sur toute la ligne : `df.apply(f, axis=1)`.

Complétez la variable `cats` ci-dessous.


In [49]:
avec_heures = ajouter_heure(connexions)
avec_heures.head(3)

,timestamp,username,ip,heure
0,2026-10-07 08:00:00,alice.dupont,192.0.2.48,8
1,2026-10-07 18:00:00,alice.dupont,192.0.2.48,18
2,2026-10-08 08:00:00,alice.dupont,192.0.2.48,8


In [52]:
cats = avec_heures["heure"].apply(categoriser_heure)
cats

0       travail
1       travail
2       travail
3       travail
4       travail
         ...   
997     travail
998     travail
999     travail
1000    travail
1001    travail
Name: heure, Length: 1002, dtype: str

In [ ]:
assert isinstance(cats, pd.Series)
assert set(cats.unique()) == {"travail", "soir"}, f"catégories : {set(cats.unique())}"
assert (cats[avec_heures["heure"] == 20] == "soir").all()
print("ex 7 OK : catégories =", sorted(cats.unique()))


## Exercice 8 — Joindre deux tables (`merge`)

La géolocalisation vit dans une table séparée (`ip_geo.csv`). Pour l'attacher
au journal, on fait une **jointure** (`merge`) sur la colonne commune `ip` :
chaque ligne du journal reçoit la latitude/longitude/pays de son IP.

- `pd.merge(gauche, droite, on="ip", how="left")` : joint sur `ip`.
- `how="left"` : garde toutes les lignes de gauche, même sans correspondance
  (les colonnes ajoutées vaudraient alors `NaN`).

Complétez la variable `df_geo` ci-dessous.

In [ ]:
df_geo = 
# TODO
pass


In [ ]:
assert "latitude" in df_geo.columns and "city" in df_geo.columns
assert df_geo.shape[1] == 7, f"attendu 7 colonnes, obtenu {df_geo.shape[1]}"
assert not df_geo["latitude"].isna().any(), "des IP sans géolocalisation"
print("ex 8 OK :", df_geo.shape[1], "colonnes,", df_geo.shape[0], "lignes")


## Exercice 9 — Grouper et agréger (`groupby` + `agg`)

Reste l'opération reine de l'analyse de logs : **regrouper** les lignes par une
clé puis **agréger** (compter, moyenner…) chaque groupe. C'est ainsi qu'on repère
les domaines les plus contactés ou les hôtes les plus bavards.

On charge maintenant `logs_reseau.csv`, le jeu de connexions sortantes du TP
« détection de beacon » :


In [ ]:
logs = pd.read_csv("logs_reseau.csv", parse_dates=["timestamp"])
logs.head()


- `df.groupby("clé")` : regroupe les lignes par valeur de `clé`.
- `.agg(n=("col", "count"), moyenne=("col", "mean"))` : agrège chaque groupe,
  une colonne de résultat par agrégation.

Complétez la fonction `trafic_par_domaine()` ci-dessous.


In [ ]:
def trafic_par_domaine(df: pd.DataFrame) -> pd.DataFrame:
    """Par domaine : nombre de connexions et taille moyenne, triés par nombre décroissant."""
    # TODO
    pass


In [ ]:
stats = trafic_par_domaine(logs)
assert stats.index.name == "domain"
assert "n" in stats.columns and "octets_moyen" in stats.columns
assert stats["n"].iloc[0] == 396, f"attendu 396 en tête, obtenu {stats['n'].iloc[0]}"
assert stats.index[0] == "av-updates.example.net"
assert stats.loc["sync-updates.example.net", "octets_moyen"] == 512.0
print("ex 9 OK :", len(stats), "domaines ; en tête :", stats.index[0])


## Exercice 10 — Combiner filtre, agrégation et sélection

En analyse de journaux, on enchaîne rarement une seule opération : on **filtre**
les lignes qui nous intéressent, on **agrège** les groupes obtenus, puis on
**sélectionne** les colonnes à conserver.

Complétez la fonction `contacts_domaine()` ci-dessous, qui enchaîne ces trois
étapes :

1. filtrer les connexions vers `domaine` (colonne `domain`) ;
2. agréger par `src_ip` : nombre de connexions (`count`), total (`sum`) et
   moyenne (`mean`) des octets ;
3. ne garder que les colonnes `connexions` et `octets`, triées par octets
   décroissants.


In [ ]:
def contacts_domaine(df: pd.DataFrame, domaine: str) -> pd.DataFrame:
    """Pour les connexions vers `domaine` : par poste source:
    - nombre de connexions,
    - total d'octets sortants
    - nombre moyen d'octets sortants
    triés par total d'octets décroissants.
    """
    # TODO
    pass


In [ ]:
res = contacts_domaine(logs, "windowsupdate.com")
assert res.index.name == "src_ip"
assert list(res.columns) == ["connexions", "octets"], f"colonnes : {list(res.columns)}"
assert (res["connexions"] == 24).all()
assert res["octets"].sum() == 21953091
assert res["octets"].iloc[0] == 2796800
print("ex 10 OK :", len(res), "postes, total =", res["octets"].sum(), "octets")


## Bilan

| notion | fonction pandas | exercice |
| --- | --- | --- |
| charger un CSV | `pd.read_csv(..., parse_dates=...)` | 1 |
| inspecter | `.head()`, `.info()`, `.shape` | 1 |
| sélectionner une colonne | `df["col"]` | 2 |
| sélectionner plusieurs colonnes | `df[["col1", "col2"]]` | 3 |
| filtrer | `df[masque_booléen]` | 4 |
| assigner une colonne | `df["nouvelle"] = ...` | 5 |
| transformer par correspondance | `.map(dico)` | 6 |
| transformer par calcul | `.apply(f)` | 7 |
| joindre | `pd.merge(..., on=...)` | 8 |
| grouper + agréger | `.groupby(...).agg(...)` | 9 |
| combiner filtre + agrégation + sélection | enchaînement des trois | 10 |

**La suite.** Ces notions sont exactement celles des deux TP qui suivent :

- **détection de beacon** : `groupby` + `diff` + `agg` pour mesurer la régularité
  des connexions vers un C2 ;
- **voyage impossible** : `merge`, `map`, `isin` et `groupby().nunique()` pour
  détecter des déplacements impossibles et écarter les faux positifs.
